CLEAN DATA TMDB

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load data
# -----------------------------
df = pd.read_csv("tmdb_titles.csv")

# -----------------------------
# 2. Remove duplicates
# -----------------------------
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["id", "content_type"])

# -----------------------------
# 3. Clean text columns
# -----------------------------
text_cols = [
    "title", "original_title", "content_type", "genre_ids", "genre_names",
    "original_language", "overview", "origin_country", "status",
    "original_name", "homepage", "imdb_id", "production_companies",
    "production_countries", "spoken_languages"
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

df = df.replace(["", "nan", "None", "null", "NaN"], np.nan)

df = df[df["title"].notna()]
df = df[df["title"].str.strip() != ""]

# -----------------------------
# 4. Convert dates and numerics
# -----------------------------
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year

numeric_cols = [
    "popularity", "vote_average", "vote_count", "runtime",
    "number_of_seasons", "number_of_episodes", "release_year"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Fix impossible values
df.loc[(df["vote_average"] < 0) | (df["vote_average"] > 10), "vote_average"] = np.nan
df.loc[df["vote_count"] < 0, "vote_count"] = np.nan
df.loc[df["popularity"] < 0, "popularity"] = np.nan

# -----------------------------
# 5. Fill missing values
# -----------------------------
df["genre_names"] = df["genre_names"].fillna("Unknown")
df["original_language"] = df["original_language"].fillna("unknown")
df["origin_country"] = df["origin_country"].fillna("Unknown")
df["overview"] = df["overview"].fillna("No overview available")
df["status"] = df["status"].fillna("Unknown")

# Numeric fills for scoring
df["popularity"] = df["popularity"].fillna(df["popularity"].median())
df["vote_average"] = df["vote_average"].fillna(df["vote_average"].median())
df["vote_count"] = df["vote_count"].fillna(0)

# -----------------------------
# 6. Feature engineering
# -----------------------------
current_year = pd.Timestamp.today().year
df["title_age"] = current_year - df["release_year"]
df["title_age"] = df["title_age"].fillna(df["title_age"].median())

df["is_recent"] = np.where(df["title_age"] <= 5, 1, 0)

def min_max_scale(series):
    #SABINA COMMENT: Why not use MinMaxScaler?
    min_val = series.min()
    max_val = series.max()
    if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
        return pd.Series(0.5, index=series.index)
    return (series - min_val) / (max_val - min_val)

df["popularity_norm"] = min_max_scale(df["popularity"])
df["vote_count_norm"] = min_max_scale(df["vote_count"])
df["vote_average_norm"] = min_max_scale(df["vote_average"])

# Freshness: more recent = higher score
if df["title_age"].max() == df["title_age"].min():
    df["freshness_score"] = 0.5
else:
    df["freshness_score"] = 1 - min_max_scale(df["title_age"])

df["visibility_score"] = df["popularity_norm"] * 100
df["engagement_score"] = df["vote_count_norm"] * 100
df["audience_reception_score"] = df["vote_average_norm"] * 100

df["business_value_score"] = (
    0.40 * df["popularity_norm"] +
    0.30 * df["vote_count_norm"] +
    0.20 * df["vote_average_norm"] +
    0.10 * df["freshness_score"]
) * 100

# -----------------------------
# 7. Segments
# -----------------------------
visibility_median = df["visibility_score"].median()
engagement_median = df["engagement_score"].median()

def segment_title(row):
    # SABINA COMMENT: Curious as to why you chose a 2-tier segmentation 
    # instead of 3-tier one?
    high_visibility = row["visibility_score"] >= visibility_median
    high_engagement = row["engagement_score"] >= engagement_median

    if high_visibility and high_engagement:
        return "High Visibility / High Engagement"
    elif high_visibility and not high_engagement:
        return "High Visibility / Low Engagement"
    elif not high_visibility and high_engagement:
        return "Low Visibility / High Engagement"
    else:
        return "Low Visibility / Low Engagement"

df["marketing_segment"] = df.apply(segment_title, axis=1)

# Hidden gems
vote_avg_threshold = df["vote_average"].quantile(0.75)
popularity_threshold = df["popularity"].quantile(0.25)

df["hidden_gem"] = np.where(
    (df["vote_average"] >= vote_avg_threshold) &
    (df["popularity"] <= popularity_threshold),
    1,
    0
)

# Tentpole content
business_threshold = df["business_value_score"].quantile(0.90)
df["tentpole_content"] = np.where(df["business_value_score"] >= business_threshold, 1, 0)

# -----------------------------
# 8. Save
# -----------------------------
output_file = "tmdb_titles_clean.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Saved:", output_file)
print("Final shape:", df.shape)
print(df.head())

Saved: tmdb_titles_clean.csv
Final shape: (13736, 40)
  source       id                      title             original_title  \
0   tmdb  1265609                War Machine                War Machine   
1   tmdb  1290821                    Shelter                    Shelter   
2   tmdb  1523145  Your Heart Will Be Broken  Твое сердце будет разбито   
3   tmdb   799882                  The Bluff                  The Bluff   
4   tmdb  1084242                 Zootopia 2                 Zootopia 2   

  content_type                  genre_ids  \
0        movie              [28, 878, 53]   
1        movie               [28, 80, 53]   
2        movie                [10749, 18]   
3        movie                   [28, 53]   
4        movie  [16, 35, 12, 10751, 9648]   

                                     genre_names original_language  \
0              Action, Science Fiction, Thriller                en   
1                        Action, Crime, Thriller                en   
2             

CLEAN DATA TV MAZE

In [22]:
import pandas as pd
import numpy as np
import re

# -----------------------------
# 1. Load data
# -----------------------------
df = pd.read_csv("tvmaze_titles.csv")

print("Original shape:", df.shape)


# -----------------------------
# 2. Basic cleaning
# -----------------------------
# Remove full duplicate rows
df = df.drop_duplicates()

# Remove duplicates by main identity
df = df.drop_duplicates(subset=["id", "content_type"])

# Replace fake null strings with real NaN
df = df.replace(["", " ", "nan", "None", "null", "NaN"], np.nan)


# -----------------------------
# 3. Clean text columns
# -----------------------------
text_cols = [
    "title",
    "content_type",
    "genre_names",
    "original_language",
    "overview",
    "origin_country",
    "status",
    "network",
    "web_channel",
    "official_site",
    "imdb_id",
    "tvdb_id",
    "source"
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace(["nan", "None", "null", ""], np.nan)


# Remove HTML tags from overview just in case
def clean_html(text):
    if pd.isna(text):
        return text
    text = re.sub(r"<[^>]+>", "", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

if "overview" in df.columns:
    df["overview"] = df["overview"].apply(clean_html)

# Remove rows with missing or empty titles
df = df[df["title"].notna()]
df = df[df["title"].str.strip() != ""]


# -----------------------------
# 4. Convert dates
# -----------------------------
if "release_date" in df.columns:
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
    df["release_year"] = df["release_date"].dt.year


# -----------------------------
# 5. Convert numerics
# -----------------------------
numeric_cols = [
    "popularity",
    "vote_average",
    "vote_count",
    "runtime",
    "average_runtime",
    "release_year"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Clean impossible values
if "vote_average" in df.columns:
    df.loc[(df["vote_average"] < 0) | (df["vote_average"] > 10), "vote_average"] = np.nan

if "vote_count" in df.columns:
    df.loc[df["vote_count"] < 0, "vote_count"] = np.nan

if "popularity" in df.columns:
    df.loc[df["popularity"] < 0, "popularity"] = np.nan

if "runtime" in df.columns:
    df.loc[df["runtime"] < 0, "runtime"] = np.nan

if "average_runtime" in df.columns:
    df.loc[df["average_runtime"] < 0, "average_runtime"] = np.nan


# -----------------------------
# 6. Standardize categories
# -----------------------------
if "content_type" in df.columns:
    df["content_type"] = df["content_type"].str.lower().replace({
        "tv show": "tv",
        "show": "tv",
        "series": "tv"
    })

# TVMaze is TV data, so if content_type is missing fill with tv
if "content_type" in df.columns:
    df["content_type"] = df["content_type"].fillna("tv")
else:
    df["content_type"] = "tv"


# -----------------------------
# 7. Handle missing values
# -----------------------------
fill_text_map = {
    "genre_names": "Unknown",
    "original_language": "unknown",
    "origin_country": "Unknown",
    "status": "Unknown",
    "network": "Unknown",
    "web_channel": "Unknown",
    "overview": "No overview available"
}

for col, value in fill_text_map.items():
    if col in df.columns:
        df[col] = df[col].fillna(value)

# Since TVMaze usually does NOT provide popularity / vote_count,
# keep them but fill carefully for modeling compatibility
if "popularity" in df.columns:
    df["popularity"] = df["popularity"].fillna(0)
else:
    df["popularity"] = 0

if "vote_count" in df.columns:
    df["vote_count"] = df["vote_count"].fillna(0)
else:
    df["vote_count"] = 0

if "vote_average" in df.columns:
    df["vote_average"] = df["vote_average"].fillna(df["vote_average"].median())
else:
    df["vote_average"] = 0


# -----------------------------
# 8. Feature engineering
# -----------------------------
current_year = pd.Timestamp.today().year

df["title_age"] = current_year - df["release_year"]
df["title_age"] = df["title_age"].replace([np.inf, -np.inf], np.nan)
df["title_age"] = df["title_age"].fillna(df["title_age"].median())

df["is_recent"] = np.where(df["title_age"] <= 5, 1, 0)


def min_max_scale(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
        return pd.Series(0.5, index=series.index)

    return (series - min_val) / (max_val - min_val)


df["popularity_norm"] = min_max_scale(df["popularity"])
df["vote_count_norm"] = min_max_scale(df["vote_count"])
df["vote_average_norm"] = min_max_scale(df["vote_average"])

# More recent = better freshness
if df["title_age"].max() == df["title_age"].min():
    df["freshness_score"] = 0.5
else:
    df["freshness_score"] = 1 - min_max_scale(df["title_age"])

# Project scores
df["visibility_score"] = df["popularity_norm"] * 100
df["engagement_score"] = df["vote_count_norm"] * 100
df["audience_reception_score"] = df["vote_average_norm"] * 100

df["business_value_score"] = (
    0.40 * df["popularity_norm"] +
    0.30 * df["vote_count_norm"] +
    0.20 * df["vote_average_norm"] +
    0.10 * df["freshness_score"]
) * 100


# -----------------------------
# 9. Marketing segmentation
# -----------------------------
visibility_median = df["visibility_score"].median()
engagement_median = df["engagement_score"].median()

def segment_title(row):
    high_visibility = row["visibility_score"] >= visibility_median
    high_engagement = row["engagement_score"] >= engagement_median

    if high_visibility and high_engagement:
        return "High Visibility / High Engagement"
    elif high_visibility and not high_engagement:
        return "High Visibility / Low Engagement"
    elif not high_visibility and high_engagement:
        return "Low Visibility / High Engagement"
    else:
        return "Low Visibility / Low Engagement"

df["marketing_segment"] = df.apply(segment_title, axis=1)

# Hidden gems:
# strong reception but low visibility
vote_avg_threshold = df["vote_average"].quantile(0.75)
visibility_threshold = df["visibility_score"].quantile(0.25)

df["hidden_gem"] = np.where(
    (df["vote_average"] >= vote_avg_threshold) &
    (df["visibility_score"] <= visibility_threshold),
    1,
    0
)

# Tentpole content:
business_threshold = df["business_value_score"].quantile(0.90)
df["tentpole_content"] = np.where(df["business_value_score"] >= business_threshold, 1, 0)


# -----------------------------
# 10. Reorder columns
# -----------------------------
preferred_order = [
    "source",
    "id",
    "title",
    "content_type",
    "genre_names",
    "original_language",
    "popularity",
    "vote_average",
    "vote_count",
    "release_date",
    "release_year",
    "overview",
    "origin_country",
    "status",
    "runtime",
    "average_runtime",
    "network",
    "web_channel",
    "official_site",
    "imdb_id",
    "tvdb_id",
    "title_age",
    "is_recent",
    "popularity_norm",
    "vote_count_norm",
    "vote_average_norm",
    "freshness_score",
    "visibility_score",
    "engagement_score",
    "audience_reception_score",
    "business_value_score",
    "marketing_segment",
    "hidden_gem",
    "tentpole_content"
]

existing_cols = [col for col in preferred_order if col in df.columns]
remaining_cols = [col for col in df.columns if col not in existing_cols]

df = df[existing_cols + remaining_cols]


# -----------------------------
# 11. Save clean dataset
# -----------------------------
output_file = "tvmaze_titles_clean.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Cleaned shape:", df.shape)
print(f"Saved cleaned file as: {output_file}")
print(df.head())

Original shape: (86460, 21)
Cleaned shape: (86460, 34)
Saved cleaned file as: tvmaze_titles_clean.csv
   source  id               title content_type  \
0  tvmaze   1      Under the Dome           tv   
1  tvmaze   2  Person of Interest           tv   
2  tvmaze   3              Bitten           tv   
3  tvmaze   4               Arrow           tv   
4  tvmaze   5      True Detective           tv   

                        genre_names original_language  popularity  \
0  Drama, Science-Fiction, Thriller           English         0.0   
1    Action, Crime, Science-Fiction           English         0.0   
2            Drama, Horror, Romance           English         0.0   
3    Drama, Action, Science-Fiction           English         0.0   
4            Drama, Crime, Thriller           English         0.0   

   vote_average  vote_count release_date  ...  vote_count_norm  \
0           6.6         0.0   2013-06-24  ...              0.5   
1           8.8         0.0   2011-09-22  ...     

MERGING BOTH DATA

In [23]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load both datasets
# -----------------------------
tmdb = pd.read_csv("tmdb_titles_clean.csv")
tvmaze = pd.read_csv("tvmaze_titles_clean.csv")

print("TMDb shape:", tmdb.shape)
print("TVMaze shape:", tvmaze.shape)


# -----------------------------
# 2. Ensure source column exists
# -----------------------------
tmdb["source"] = "tmdb"
tvmaze["source"] = "tvmaze"


# -----------------------------
# 3. Align columns (VERY IMPORTANT)
# -----------------------------
# Get all columns from both datasets
all_columns = list(set(tmdb.columns).union(set(tvmaze.columns)))

# Add missing columns to each dataset
for col in all_columns:
    if col not in tmdb.columns:
        tmdb[col] = np.nan
    if col not in tvmaze.columns:
        tvmaze[col] = np.nan

# Ensure same column order
tmdb = tmdb[all_columns]
tvmaze = tvmaze[all_columns]


# -----------------------------
# 4. Combine datasets
# -----------------------------
df = pd.concat([tmdb, tvmaze], ignore_index=True)

print("Combined shape:", df.shape)


# -----------------------------
# 5. Clean after merge
# -----------------------------
# Remove duplicates (same title + year + type)
df = df.drop_duplicates(subset=["title", "release_year", "content_type"])

# Clean title again just in case
df = df[df["title"].notna()]
df = df[df["title"].str.strip() != ""]

# Reset index
df = df.reset_index(drop=True)

# -----------------------------
# Language normalization mapping
# -----------------------------

language_map = {
    "en": "English",
    "fr": "French",
    "es": "Spanish",
    "de": "German",
    "it": "Italian",
    "pt": "Portuguese",
    "ru": "Russian",
    "ja": "Japanese",
    "zh": "Chinese",
    "ko": "Korean",
    "ar": "Arabic",
    "hi": "Hindi",
    "tr": "Turkish",
    "nl": "Dutch",
    "sv": "Swedish",
    "no": "Norwegian",
    "da": "Danish",
    "fi": "Finnish",
    "pl": "Polish",
    "cs": "Czech",
    "el": "Greek",
    "he": "Hebrew",
    "th": "Thai",
    "id": "Indonesian",
    "ms": "Malay",
    "vi": "Vietnamese",
    "uk": "Ukrainian",
    "hu": "Hungarian",
    "ro": "Romanian",
    "sk": "Slovak",
    "bg": "Bulgarian",
    "hr": "Croatian",
    "sr": "Serbian",
    "lt": "Lithuanian",
    "lv": "Latvian",
    "et": "Estonian",
    "fa": "Persian",
    "bn": "Bengali",
    "ta": "Tamil",
    "te": "Telugu",
    "ml": "Malayalam",
    "kn": "Kannada",
    "mr": "Marathi",
    "ur": "Urdu",
    "pa": "Punjabi",

    "en": "English",
    "uk": "English (UK)",
    "fr": "French",
    "es": "Spanish",
    "de": "German",
    "it": "Italian",
    "pt": "Portuguese",
    "ru": "Russian",
    "ja": "Japanese",
    "zh": "Chinese",
    "ko": "Korean",

    # Fix abbreviations from your list
    "af": "Afrikaans",
    "tl": "Tagalog",
    "ka": "Georgian",
    "bs": "Bosnian",
    "mn": "Mongolian",
    "as": "Assamese",
    "is": "Icelandic",
    "si": "Sinhalese",
    "ab": "Abkhazian",
    "ga": "Irish",
    "eu": "Basque",
    "ne": "Nepali",
    "ca": "Catalan",
    "az": "Azerbaijani",
    "cy": "Welsh",
    "sq": "Albanian",
    "nb": "Norwegian (Bokmål)",
    "sl": "Slovenian",
    "gl": "Galician",
    "cr": "Cree",
    "sh": "Serbo-Croatian",

    # Fix duplicates / spelling
    "panjabi": "Punjabi",
    "punjabi": "Punjabi",
    "scottish gaelic": "Scottish Gaelic",

    # Already full but normalize casing
    "english": "English",
    "french": "French",
    "german": "German",
    "spanish": "Spanish",
    "arabic": "Arabic",
    "hindi": "Hindi",
    "turkish": "Turkish",
    "persian": "Persian",
}

def clean_language(lang):
    if pd.isna(lang):
        return "Unknown"
    
    lang = str(lang).strip().lower()
    
    if lang in ["unknown", "xx", "cn"]:
        return "Unknown"
    
    if lang in language_map:
        return language_map[lang]
    
    return lang.capitalize()


df["original_language"] = df["original_language"].apply(clean_language)

# -----------------------------
# Apply to dataframe
# -----------------------------
df["original_language_clean"] = df["original_language"].apply(clean_language)


# -----------------------------
# 6. Optional: create unified scores (recommended)
# -----------------------------
# Since TVMaze had weaker metrics, recompute global normalized scores

def min_max_scale(series):
    min_val = series.min()
    max_val = series.max()
    if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
        return pd.Series(0.5, index=series.index)
    return (series - min_val) / (max_val - min_val)

# Fill missing numeric values safely
df["popularity"] = df["popularity"].fillna(df["popularity"].median())
df["vote_count"] = df["vote_count"].fillna(0)
df["vote_average"] = df["vote_average"].fillna(df["vote_average"].median())

# Normalize across ALL data
df["popularity_norm"] = min_max_scale(df["popularity"])
df["vote_count_norm"] = min_max_scale(df["vote_count"])
df["vote_average_norm"] = min_max_scale(df["vote_average"])

# Freshness
current_year = pd.Timestamp.today().year
df["title_age"] = current_year - df["release_year"]
df["title_age"] = df["title_age"].fillna(df["title_age"].median())

df["freshness_score"] = 1 - min_max_scale(df["title_age"])

# Final Business Score
df["business_value_score"] = (
    0.40 * df["popularity_norm"] +
    0.30 * df["vote_count_norm"] +
    0.20 * df["vote_average_norm"] +
    0.10 * df["freshness_score"]
) * 100


# -----------------------------
# 7. Final sorting (optional)
# -----------------------------
df = df.sort_values(by="business_value_score", ascending=False)


# -----------------------------
# 8. Save final dataset
# -----------------------------
output_file = "all_streaming_titles.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\nFinal dataset saved:", output_file)
print("Final shape:", df.shape)
print(df.head())

TMDb shape: (13736, 40)
TVMaze shape: (86460, 34)
Combined shape: (100196, 45)

Final dataset saved: all_streaming_titles.csv
Final shape: (96200, 46)
      popularity_norm                                           homepage  \
7554         0.916136  https://abc.com/show/ada4e644-ce0b-4590-9dac-b...   
0            1.000000             https://www.netflix.com/title/81768525   
7558         0.796415  https://abc.com/show/b7d90f97-dbb2-4ab5-b141-f...   
7556         0.830130         https://www.warnerbros.com/tv/supernatural   
7587         0.445930                https://www.hbo.com/game-of-thrones   

      engagement_score release_date  popularity       id adult  \
7554          7.789985   2018-10-16    385.1942    79744   0.0   
0             2.340317   2026-02-12    420.4551  1265609   0.0   
7558         27.603475   2005-03-27    334.8569     1416   0.0   
7556         21.134389   2005-09-13    349.0323     1622   0.0   
7587         67.621359   2011-04-17    187.4934     1399   0.0

In [24]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load both datasets
# -----------------------------
tmdb = pd.read_csv("tmdb_titles_clean.csv")
tvmaze = pd.read_csv("tvmaze_titles_clean.csv")

print("TMDb shape:", tmdb.shape)
print("TVMaze shape:", tvmaze.shape)


# -----------------------------
# 2. Ensure source column exists
# -----------------------------
tmdb["source"] = "tmdb"
tvmaze["source"] = "tvmaze"


# -----------------------------
# 3. Align columns
# -----------------------------
all_columns = list(set(tmdb.columns).union(set(tvmaze.columns)))

for col in all_columns:
    if col not in tmdb.columns:
        tmdb[col] = np.nan
    if col not in tvmaze.columns:
        tvmaze[col] = np.nan

tmdb = tmdb[all_columns]
tvmaze = tvmaze[all_columns]


# -----------------------------
# 4. Combine datasets
# -----------------------------
df = pd.concat([tmdb, tvmaze], ignore_index=True)

print("Combined shape:", df.shape)


# -----------------------------
# 5. Clean after merge
# -----------------------------
# Normalize content type first
if "content_type" in df.columns:
    df["content_type"] = df["content_type"].astype(str).str.strip().str.lower()

# Clean title
df = df[df["title"].notna()]
df["title"] = df["title"].astype(str).str.strip()
df = df[df["title"] != ""]

# Remove duplicates
df = df.drop_duplicates(subset=["title", "release_year", "content_type"])

# Reset index
df = df.reset_index(drop=True)


# -----------------------------
# 5.1 Language normalization
# -----------------------------
language_map = {
    # ISO codes
    "en": "English",
    "fr": "French",
    "es": "Spanish",
    "de": "German",
    "it": "Italian",
    "pt": "Portuguese",
    "ru": "Russian",
    "ja": "Japanese",
    "zh": "Chinese",
    "ko": "Korean",
    "ar": "Arabic",
    "hi": "Hindi",
    "tr": "Turkish",
    "nl": "Dutch",
    "sv": "Swedish",
    "no": "Norwegian",
    "nb": "Norwegian (Bokmål)",
    "da": "Danish",
    "fi": "Finnish",
    "pl": "Polish",
    "cs": "Czech",
    "el": "Greek",
    "he": "Hebrew",
    "th": "Thai",
    "id": "Indonesian",
    "ms": "Malay",
    "vi": "Vietnamese",
    "uk": "Ukrainian",
    "hu": "Hungarian",
    "ro": "Romanian",
    "sk": "Slovak",
    "bg": "Bulgarian",
    "hr": "Croatian",
    "sr": "Serbian",
    "lt": "Lithuanian",
    "lv": "Latvian",
    "et": "Estonian",
    "fa": "Persian",
    "bn": "Bengali",
    "ta": "Tamil",
    "te": "Telugu",
    "ml": "Malayalam",
    "kn": "Kannada",
    "mr": "Marathi",
    "ur": "Urdu",
    "pa": "Punjabi",
    "af": "Afrikaans",
    "tl": "Tagalog",
    "ka": "Georgian",
    "bs": "Bosnian",
    "mn": "Mongolian",
    "as": "Assamese",
    "is": "Icelandic",
    "si": "Sinhalese",
    "ab": "Abkhazian",
    "ga": "Irish",
    "eu": "Basque",
    "ne": "Nepali",
    "ca": "Catalan",
    "az": "Azerbaijani",
    "cy": "Welsh",
    "sq": "Albanian",
    "sl": "Slovenian",
    "gl": "Galician",
    "cr": "Cree",
    "sh": "Serbo-Croatian",
    "kk": "Kazakh",
    "ky": "Kyrgyz",
    "sw": "Swahili",
    "gu": "Gujarati",
    "lo": "Lao",
    "ps": "Pashto",
    "mg": "Malagasy",
    "yo": "Yoruba",
    "fj": "Fijian",
    "hy": "Armenian",
    "be": "Belarusian",
    "my": "Burmese",
    "jv": "Javanese",
    "wo": "Wolof",
    "dv": "Divehi",
    "kg": "Kongo",
    "la": "Latin",
    "ce": "Chechen",
    "zu": "Zulu",

    # Full names / spelling fixes
    "english": "English",
    "french": "French",
    "german": "German",
    "spanish": "Spanish",
    "japanese": "Japanese",
    "chinese": "Chinese",
    "korean": "Korean",
    "portuguese": "Portuguese",
    "arabic": "Arabic",
    "hindi": "Hindi",
    "turkish": "Turkish",
    "persian": "Persian",
    "russian": "Russian",
    "italian": "Italian",
    "dutch": "Dutch",
    "swedish": "Swedish",
    "norwegian": "Norwegian",
    "danish": "Danish",
    "polish": "Polish",
    "finnish": "Finnish",
    "hebrew": "Hebrew",
    "hungarian": "Hungarian",
    "romanian": "Romanian",
    "czech": "Czech",
    "bulgarian": "Bulgarian",
    "croatian": "Croatian",
    "serbian": "Serbian",
    "lithuanian": "Lithuanian",
    "latvian": "Latvian",
    "estonian": "Estonian",
    "bengali": "Bengali",
    "tamil": "Tamil",
    "telugu": "Telugu",
    "malayalam": "Malayalam",
    "kannada": "Kannada",
    "marathi": "Marathi",
    "urdu": "Urdu",
    "punjabi": "Punjabi",
    "panjabi": "Punjabi",
    "afrikaans": "Afrikaans",
    "tagalog": "Tagalog",
    "georgian": "Georgian",
    "bosnian": "Bosnian",
    "mongolian": "Mongolian",
    "assamese": "Assamese",
    "icelandic": "Icelandic",
    "sinhalese": "Sinhalese",
    "irish": "Irish",
    "basque": "Basque",
    "nepali": "Nepali",
    "catalan": "Catalan",
    "azerbaijani": "Azerbaijani",
    "welsh": "Welsh",
    "albanian": "Albanian",
    "slovenian": "Slovenian",
    "galician": "Galician",
    "kazakh": "Kazakh",
    "uzbek": "Uzbek",
    "zulu": "Zulu",
    "scottish gaelic": "Scottish Gaelic",
    "burmese": "Burmese",
    "armenian": "Armenian",
    "yoruba": "Yoruba",
    "swahili": "Swahili",
    "kyrgyz": "Kyrgyz",
    "gujarati": "Gujarati",
    "lao": "Lao",
    "wolof": "Wolof",
    "divehi": "Divehi",
    "malagasy": "Malagasy",
    "kongo": "Kongo",
    "latin": "Latin",
    "chechen": "Chechen",
    "abkhazian": "Abkhazian",
    "ukrainian": "Ukrainian",
    "vietnamese": "Vietnamese",
    "indonesian": "Indonesian",
    "malay": "Malay",
    "greek": "Greek",
    "thai": "Thai",
    "luxembourgish": "Luxembourgish"
}

def clean_language(lang):
    if pd.isna(lang):
        return "Unknown"

    lang = str(lang).strip().lower()

    if lang in ["", "unknown", "xx", "cn", "null", "none", "nan"]:
        return "Unknown"

    if lang in language_map:
        return language_map[lang]

    return lang.title()

if "original_language" in df.columns:
    df["original_language"] = df["original_language"].apply(clean_language)


# -----------------------------
# 5.2 Clean text columns
# -----------------------------
if "overview" in df.columns:
    df["overview"] = df["overview"].fillna("").astype(str).str.strip()

if "original_title" in df.columns:
    df["original_title"] = df["original_title"].fillna("").astype(str).str.strip()


# -----------------------------
# 5.3 Runtime cleanup
# -----------------------------
if "runtime" in df.columns or "average_runtime" in df.columns:
    if "runtime" not in df.columns:
        df["runtime"] = np.nan
    if "average_runtime" not in df.columns:
        df["average_runtime"] = np.nan

    df["runtime_final"] = df["runtime"].fillna(df["average_runtime"])

    # Remove impossible values, but keep NaN
    df.loc[df["runtime_final"] <= 0, "runtime_final"] = np.nan
    df.loc[df["runtime_final"] > 300, "runtime_final"] = np.nan


# -----------------------------
# 5.4 Drop very empty columns
# Keep Unknown values in data, only drop columns if too empty
# -----------------------------
missing_ratio = df.isna().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.92].index.tolist()

# Protect useful columns
protected_cols = {
    "title", "release_year", "content_type", "original_language",
    "business_value_score", "popularity", "vote_count", "vote_average",
    "runtime_final", "source", "overview"
}

cols_to_drop = [col for col in cols_to_drop if col not in protected_cols]

df = df.drop(columns=cols_to_drop, errors="ignore")


# -----------------------------
# 6. Create unified scores
# -----------------------------
def min_max_scale(series):
    min_val = series.min()
    max_val = series.max()
    if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
        return pd.Series(0.5, index=series.index)
    return (series - min_val) / (max_val - min_val)

# Safe numeric filling
if "popularity" in df.columns:
    df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce")
    df["popularity"] = df["popularity"].fillna(df["popularity"].median())

if "vote_count" in df.columns:
    df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce")
    df["vote_count"] = df["vote_count"].fillna(0)

if "vote_average" in df.columns:
    df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce")
    df["vote_average"] = df["vote_average"].fillna(df["vote_average"].median())

if "release_year" in df.columns:
    df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")

# Normalize across all data
df["popularity_norm"] = min_max_scale(df["popularity"])
df["vote_count_norm"] = min_max_scale(df["vote_count"])
df["vote_average_norm"] = min_max_scale(df["vote_average"])

# Freshness
current_year = pd.Timestamp.today().year
df["title_age"] = current_year - df["release_year"]
df["title_age"] = df["title_age"].fillna(df["title_age"].median())

df["freshness_score"] = 1 - min_max_scale(df["title_age"])

# Final Business Score
df["business_value_score"] = (
    0.40 * df["popularity_norm"] +
    0.30 * df["vote_count_norm"] +
    0.20 * df["vote_average_norm"] +
    0.10 * df["freshness_score"]
) * 100


# -----------------------------
# 7. Final sorting
# -----------------------------
df = df.sort_values(by="business_value_score", ascending=False)


# -----------------------------
# 8. Optional cleanup of redundant columns
# -----------------------------
# Remove original runtime columns if runtime_final exists
if "runtime_final" in df.columns:
    df = df.drop(columns=["runtime", "average_runtime"], errors="ignore")

# Reset index after all cleaning
df = df.reset_index(drop=True)


# -----------------------------
# 9. Save final dataset
# -----------------------------
output_file = "all_streaming_titles.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\nFinal dataset saved:", output_file)
print("Final shape:", df.shape)
print("\nUnique original_language values:")
print(sorted(df["original_language"].dropna().unique())[:100])
print("\nPreview:")
print(df.head())

TMDb shape: (13736, 40)
TVMaze shape: (86460, 34)
Combined shape: (100196, 45)

Final dataset saved: all_streaming_titles.csv
Final shape: (96200, 40)

Unique original_language values:
['Abkhazian', 'Afrikaans', 'Albanian', 'Arabic', 'Armenian', 'Assamese', 'Azerbaijani', 'Basque', 'Belarusian', 'Bengali', 'Bosnian', 'Bulgarian', 'Burmese', 'Catalan', 'Chechen', 'Chinese', 'Cree', 'Croatian', 'Czech', 'Danish', 'Divehi', 'Dutch', 'English', 'Estonian', 'Fijian', 'Finnish', 'French', 'Galician', 'Georgian', 'German', 'Greek', 'Gujarati', 'Hebrew', 'Hindi', 'Hungarian', 'Icelandic', 'Indonesian', 'Irish', 'Italian', 'Japanese', 'Javanese', 'Kannada', 'Kazakh', 'Kongo', 'Korean', 'Kyrgyz', 'Lao', 'Latin', 'Latvian', 'Lithuanian', 'Luxembourgish', 'Malagasy', 'Malay', 'Malayalam', 'Marathi', 'Mongolian', 'Nepali', 'Norwegian', 'Norwegian (Bokmål)', 'Pashto', 'Persian', 'Polish', 'Portuguese', 'Punjabi', 'Romanian', 'Russian', 'Scottish Gaelic', 'Serbian', 'Serbo-Croatian', 'Sinhalese', 'Sl

CHECK FINAL DATA 

In [25]:
import pandas as pd
import numpy as np

Clean_data = pd.read_csv ("all_streaming_titles.csv")

#Clean_data.shape
print (Clean_data.shape)
Clean_data.head

Clean_data["production_countries"].unique()
Clean_data["genre_names"].unique()
Clean_data["original_language"].unique()




(96200, 40)


array(['English', 'German', 'Spanish', 'Japanese', 'Chinese', 'Korean',
       'Portuguese', 'French', 'Afrikaans', 'Finnish', 'Serbian',
       'Unknown', 'Telugu', 'Hindi', 'Italian', 'Lithuanian', 'Turkish',
       'Russian', 'Slovak', 'Bengali', 'Danish', 'Hungarian', 'Malayalam',
       'Tagalog', 'Thai', 'Tamil', 'Arabic', 'Dutch', 'Latvian', 'Hebrew',
       'Swedish', 'Greek', 'Norwegian', 'Romanian', 'Malay', 'Czech',
       'Indonesian', 'Polish', 'Kannada', 'Georgian', 'Bosnian',
       'Mongolian', 'Assamese', 'Persian', 'Icelandic', 'Sinhalese',
       'Urdu', 'Abkhazian', 'Vietnamese', 'Ukrainian', 'Estonian',
       'Marathi', 'Bulgarian', 'Irish', 'Welsh', 'Basque', 'Nepali',
       'Luxembourgish', 'Croatian', 'Catalan', 'Kazakh', 'Punjabi',
       'Uzbek', 'Zulu', 'Scottish Gaelic', 'Burmese', 'Belarusian',
       'Armenian', 'Fijian', 'Yoruba', 'Azerbaijani', 'Swahili', 'Kyrgyz',
       'Javanese', 'Slovenian', 'Gujarati', 'Lao', 'Galician', 'Wolof',
       'Divehi',